In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error

from catboost import CatBoostRegressor

In [2]:
TRAIN_PATH = "/kaggle/input/datasets/rohanthakar/traffic-demand-prediction-gridlock/dataset/train.csv"
TEST_PATH = "/kaggle/input/datasets/rohanthakar/traffic-demand-prediction-gridlock/dataset/test.csv"
SAMPLE_PATH = "/kaggle/input/datasets/rohanthakar/traffic-demand-prediction-gridlock/dataset/sample_submission.csv"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample = pd.read_csv(SAMPLE_PATH)

print(train.shape)
print(test.shape)

(77299, 11)
(41778, 10)


In [3]:
def create_features(df):

    df = df.copy()

    df["hour"] = df["timestamp"].str.split(":").str[0].astype(int)
    df["minute"] = df["timestamp"].str.split(":").str[1].astype(int)

    df["time_slot"] = df["hour"] * 4 + (df["minute"] // 15)

    df["is_morning_peak"] = (
        ((df["hour"] >= 7) & (df["hour"] <= 10))
    ).astype(int)

    df["is_evening_peak"] = (
        ((df["hour"] >= 16) & (df["hour"] <= 20))
    ).astype(int)

    df["is_night"] = (
        ((df["hour"] >= 22) | (df["hour"] <= 5))
    ).astype(int)

    return df

train = create_features(train)
test = create_features(test)

In [4]:
combined = pd.concat(
    [
        train.drop(columns=["demand"]),
        test
    ],
    axis=0
)

road_mode = combined["RoadType"].mode()[0]
weather_mode = combined["Weather"].mode()[0]

train["RoadType"] = train["RoadType"].fillna(road_mode)
test["RoadType"] = test["RoadType"].fillna(road_mode)

train["Weather"] = train["Weather"].fillna(weather_mode)
test["Weather"] = test["Weather"].fillna(weather_mode)

temp_median = combined["Temperature"].median()

train["Temperature"] = train["Temperature"].fillna(temp_median)
test["Temperature"] = test["Temperature"].fillna(temp_median)

In [5]:
global_mean = train["demand"].mean()

geo_mean = train.groupby("geohash")["demand"].mean()

slot_mean = train.groupby("time_slot")["demand"].mean()

geo_slot_mean = train.groupby(
    ["geohash", "time_slot"]
)["demand"].mean()

train["geo_mean"] = train["geohash"].map(geo_mean)
test["geo_mean"] = test["geohash"].map(geo_mean)

train["slot_mean"] = train["time_slot"].map(slot_mean)
test["slot_mean"] = test["time_slot"].map(slot_mean)

train["geo_slot_mean"] = train.set_index(
    ["geohash", "time_slot"]
).index.map(geo_slot_mean)

test["geo_slot_mean"] = test.set_index(
    ["geohash", "time_slot"]
).index.map(geo_slot_mean)

train["geo_slot_mean"] = train["geo_slot_mean"].fillna(global_mean)
test["geo_slot_mean"] = test["geo_slot_mean"].fillna(global_mean)

In [6]:
for df in [train, test]:

    df["lane_temp"] = (
        df["NumberofLanes"] * df["Temperature"]
    )

    df["geo_hour"] = (
        df["geohash"] + "_" + df["hour"].astype(str)
    )

    df["road_weather"] = (
        df["RoadType"] + "_" + df["Weather"]
    )

In [7]:
TARGET = "demand"

FEATURES = [
    "geohash",
    "RoadType",
    "NumberofLanes",
    "LargeVehicles",
    "Landmarks",
    "Temperature",
    "Weather",
    "day",
    "hour",
    "minute",
    "time_slot",
    "is_morning_peak",
    "is_evening_peak",
    "is_night",
    "geo_mean",
    "slot_mean",
    "geo_slot_mean",
    "lane_temp",
    "geo_hour",
    "road_weather"
]

cat_features = [
    FEATURES.index("geohash"),
    FEATURES.index("RoadType"),
    FEATURES.index("LargeVehicles"),
    FEATURES.index("Landmarks"),
    FEATURES.index("Weather"),
    FEATURES.index("geo_hour"),
    FEATURES.index("road_weather")
]

In [8]:
X = train[FEATURES]
y = train[TARGET]

groups = train["geohash"]

gkf = GroupKFold(n_splits=5)

scores = []

for fold, (tr_idx, val_idx) in enumerate(
    gkf.split(X, y, groups)
):

    X_train = X.iloc[tr_idx]
    y_train = y.iloc[tr_idx]

    X_val = X.iloc[val_idx]
    y_val = y.iloc[val_idx]

    model = CatBoostRegressor(
        iterations=5000,
        depth=8,
        learning_rate=0.03,
        loss_function="MAE",
        eval_metric="MAE",
        random_seed=42,
        verbose=500
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features,
        eval_set=(X_val, y_val),
        use_best_model=True
    )

    pred = model.predict(X_val)

    mae = mean_absolute_error(y_val, pred)

    scores.append(mae)

    print(f"Fold {fold+1} MAE = {mae}")

print("CV MAE =", np.mean(scores))

0:	learn: 0.0750538	test: 0.0623748	best: 0.0623748 (0)	total: 145ms	remaining: 12m 4s
500:	learn: 0.0043908	test: 0.0046449	best: 0.0046449 (500)	total: 34.1s	remaining: 5m 6s
1000:	learn: 0.0039141	test: 0.0045158	best: 0.0045123 (989)	total: 1m 8s	remaining: 4m 33s
1500:	learn: 0.0036607	test: 0.0044205	best: 0.0044194 (1481)	total: 1m 42s	remaining: 3m 59s
2000:	learn: 0.0035170	test: 0.0045206	best: 0.0044178 (1511)	total: 2m 16s	remaining: 3m 24s
2500:	learn: 0.0034063	test: 0.0045237	best: 0.0044178 (1511)	total: 2m 49s	remaining: 2m 49s
3000:	learn: 0.0032861	test: 0.0045284	best: 0.0044178 (1511)	total: 3m 22s	remaining: 2m 14s
3500:	learn: 0.0032047	test: 0.0045299	best: 0.0044178 (1511)	total: 3m 55s	remaining: 1m 40s
4000:	learn: 0.0031424	test: 0.0045505	best: 0.0044178 (1511)	total: 4m 27s	remaining: 1m 6s
4500:	learn: 0.0030936	test: 0.0045639	best: 0.0044178 (1511)	total: 4m 59s	remaining: 33.2s
4999:	learn: 0.0030420	test: 0.0045931	best: 0.0044178 (1511)	total: 5m 32s

In [9]:
final_model = CatBoostRegressor(
    iterations=5000,
    depth=8,
    learning_rate=0.03,
    loss_function="MAE",
    eval_metric="MAE",
    random_seed=42,
    verbose=500
)

final_model.fit(
    train[FEATURES],
    train[TARGET],
    cat_features=cat_features
)

0:	learn: 0.0721346	total: 69.2ms	remaining: 5m 45s
500:	learn: 0.0043098	total: 34.4s	remaining: 5m 8s
1000:	learn: 0.0038784	total: 1m 9s	remaining: 4m 38s
1500:	learn: 0.0036497	total: 1m 45s	remaining: 4m 6s
2000:	learn: 0.0034929	total: 2m 21s	remaining: 3m 32s
2500:	learn: 0.0033754	total: 2m 58s	remaining: 2m 58s
3000:	learn: 0.0032977	total: 3m 34s	remaining: 2m 23s
3500:	learn: 0.0032430	total: 4m 10s	remaining: 1m 47s
4000:	learn: 0.0031880	total: 4m 47s	remaining: 1m 11s
4500:	learn: 0.0031330	total: 5m 23s	remaining: 35.9s
4999:	learn: 0.0030932	total: 6m	remaining: 0us


CatBoostRegressor(depth=8, eval_metric='MAE', iterations=5000, learning_rate=0.03, loss_function='MAE', random_seed=42, verbose=500)

In [10]:
test_pred = final_model.predict(
    test[FEATURES]
)

test_pred = np.clip(
    test_pred,
    0,
    1
)

In [15]:
submission = pd.DataFrame({
    "Index": test["Index"],
    "demand": test_pred
})

print(submission.shape)

submission.to_csv(
    "submission.csv",
    index=False
)

submission.head()

(41778, 2)


,Index,demand
0,0,0.086988
1,1,0.090317
2,2,0.087559
3,3,0.063860
4,4,0.109182


In [16]:
submission.info()

submission.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41778 entries, 0 to 41777
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Index   41778 non-null  int64  
 1   demand  41778 non-null  float64
dtypes: float64(1), int64(1)
memory usage: 652.9 KB


Index     0
demand    0
dtype: int64

In [17]:
print(submission["demand"].min())
print(submission["demand"].max())

0.001125169841495452
1.0


In [19]:
submission["demand"] = submission["demand"].clip(0, 1)

submission.to_csv(
    "submission.csv",
    index=False
)

print("Saved Successfully")

Saved Successfully


In [20]:
print(train.columns.tolist())

print(train.shape)

train.head()

['Index', 'geohash', 'day', 'timestamp', 'demand', 'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather', 'hour', 'minute', 'time_slot', 'is_morning_peak', 'is_evening_peak', 'is_night', 'geo_mean', 'slot_mean', 'geo_slot_mean', 'lane_temp', 'geo_hour', 'road_weather']
(77299, 23)


,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,...,time_slot,is_morning_peak,is_evening_peak,is_night,geo_mean,slot_mean,geo_slot_mean,lane_temp,geo_hour,road_weather
0,0,qp02z1,48,0:0,0.048804,Residential,1,Not Allowed,No,16.415162,...,0,0,0,1,0.040048,0.081056,0.038157,16.415162,qp02z1_0,Residential_Sunny
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,...,0,0,0,1,0.208766,0.081056,0.114096,93.313694,qp02zt_0,Residential_Sunny
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,...,0,0,0,1,0.127931,0.081056,0.027132,25.919267,qp08bj_0,Residential_Sunny
3,3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,16.415162,...,0,0,0,1,0.014381,0.081056,0.003272,16.415162,qp08gt_0,Residential_Rainy
4,4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,...,0,0,0,1,0.029300,0.081056,0.010819,10.803667,qp02zq_0,Residential_Rainy


In [21]:
print(train["day"].value_counts().sort_index())

day
48    69427
49     7872
Name: count, dtype: int64


In [22]:
print(np.mean(scores))

0.004986398818321566


In [23]:
print(test["day"].value_counts().sort_index())

day
49    41778
Name: count, dtype: int64


In [24]:
print(
    train.groupby("day")["demand"].describe()
)

       count      mean       std           min       25%       50%       75%  \
day                                                                            
48   69427.0  0.092659  0.141837  6.245650e-07  0.017904  0.046806  0.106439   
49    7872.0  0.105262  0.144792  7.981559e-07  0.021368  0.055901  0.127472   

     max  
day       
48   1.0  
49   1.0  


In [25]:
print(
    test["day"].value_counts()
)

day
49    41778
Name: count, dtype: int64


In [26]:
train48 = train[train["day"] == 48].copy()

lookup = train48.groupby(
    ["geohash", "time_slot"]
)["demand"].mean()

test_lookup = test.set_index(
    ["geohash", "time_slot"]
).index.map(lookup)

coverage = test_lookup.notna().mean()

print("Coverage:", coverage)

Coverage: 0.8888888888888888


In [27]:
train48 = train[train["day"] == 48]

print(
    train48.groupby(
        ["geohash","time_slot"]
    ).size().describe()
)

count    69427.0
mean         1.0
std          0.0
min          1.0
25%          1.0
50%          1.0
75%          1.0
max          1.0
dtype: float64
